# Day A4 — 함수, 제어문, apply 계열, 오류 다루기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/A-r-bioconductor/day-A4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/A-r-bioconductor/day-A4.md)

## 예시로 이해하기

### 예시 1 — 입력을 검사하는 정규화 함수

In [ ]:
log2_norm <- function(counts, size_factor, pseudo = 1) {
  if (length(size_factor) != ncol(counts)) stop("size_factor 길이가 샘플 수와 다릅니다")
  log2(sweep(counts, 2, size_factor, "/") + pseudo)
}
m <- matrix(c(100, 400, 200, 800, 50, 200), nrow = 2,
            dimnames = list(c("Gfap", "Cd68"), c("S1", "S2", "S3")))
round(log2_norm(m, size_factor = c(1, 2, 0.5)), 2)
round(log2_norm(m, c(1, 2, 0.5), pseudo = 0.5), 2)
res <- tryCatch(log2_norm(m, c(1, 2)), error = function(e) conditionMessage(e))
res

### 예시 2 — closure와 제어문

In [ ]:
make_filter <- function(min_count) {
  function(x) x[x >= min_count]       # min_count를 기억하는 함수(closure)
}
keep20 <- make_filter(20)
keep100 <- make_filter(100)
v <- c(Gfap = 350, Il6 = 12, Cd68 = 80, Neg1 = 5)
keep20(v)
keep100(v)
classify <- function(lfc) {
  if (is.na(lfc)) return("결측")
  if (lfc >= 1) "증가" else if (lfc <= -1) "감소" else "변화 작음"
}
classify(1.4); classify(NA)
lfcs <- c(1.4, -0.2, -1.8)
ifelse(lfcs >= 1, "증가", ifelse(lfcs <= -1, "감소", "변화 작음"))
unit <- "log2"
switch(unit, log2 = "두 배 = 1", log10 = "열 배 = 1", "모르는 단위")

### 예시 3 — apply 계열을 한 번에 비교

In [ ]:
set.seed(3)
m <- matrix(round(2^rnorm(15, mean = 7)), nrow = 5,
            dimnames = list(paste0("g", 1:5), c("S1", "S2", "S3")))
apply(m, 1, median)                       # 1 = 행(유전자)마다
apply(m, 2, sum)                          # 2 = 열(샘플)마다
colSums(m)                                # 같은 일을 하는 전용 함수
per_sample <- lapply(colnames(m), function(s) summary(m[, s]))
names(per_sample) <- colnames(m)
per_sample$S2
sapply(colnames(m), function(s) max(m[, s]))
vapply(colnames(m), function(s) max(m[, s]), numeric(1))
value <- c(5.1, 5.4, 6.8, 7.0, 4.9, 7.3)
group <- c("ctrl", "ctrl", "treat", "treat", "ctrl", "treat")
tapply(value, group, mean)
Reduce(`+`, list(1:3, 4:6, 7:9))
Filter(function(x) x > 6, value)
Map(function(g, v) paste(g, v), c("a", "b"), c(1, 2))

### 예시 4 — 오류·경고 잡기, 뒷정리, 시간 재기

In [ ]:
safe_log2 <- function(x) {
  withCallingHandlers(
    tryCatch(log2(x),
             error = function(e) { message("오류를 잡았다: ", conditionMessage(e)); NA }),
    warning = function(w) { message("경고를 기록했다: ", conditionMessage(w)); invokeRestart("muffleWarning") })
}
safe_log2(8)
safe_log2(-1)
safe_log2("Gfap")
read_with_cleanup <- function() {
  con <- file(tempfile(), "w")
  on.exit({ close(con); message("파일 연결을 닫았다") })
  writeLines("S1,ctrl", con)
  stop("중간에 실패")
}
try(read_with_cleanup(), silent = TRUE)
x <- runif(1e6)
t_loop <- system.time({ s <- 0; for (v in x) s <- s + v^2 })["elapsed"]
t_vec  <- system.time(sum(x^2))["elapsed"]
c(loop = unname(t_loop), vectorized = unname(t_vec))

## 직접 해보기

1. `notebooks/advanced/A/A4_functions.R`을 만들고 예시 1–4를 실행합니다.
2. **z-score 함수.** 행렬을 받아 유전자(행)마다 평균을 빼고 표준편차로 나누는 `row_z <- function(m, center = mean, scale = sd)`를 만듭니다. `apply(m, 1, ...)`의 결과가 **뒤집혀**(샘플 × 유전자) 나온다는 점을 확인하고 `t()`로 바로잡습니다. 그다음 `center = median, scale = mad`로 바꿔 호출해 봅니다. 함수를 인자로 넘길 수 있다는 것이 오늘의 요점입니다.
3. **입력 검사 추가.** `row_z`의 첫 줄에 `stopifnot(is.matrix(m), is.numeric(m))`를 넣고, data.frame이나 문자 행렬을 넣었을 때 멈추는지 확인합니다.
4. **유전자마다 검정, 실패는 기록.** 예시 3의 `m`에 행 하나를 `NA`로만 채운 유전자를 추가하고, `sapply(rownames(m), function(g) tryCatch(t.test(m[g, ])$p.value, error = function(e) NA))`로 돌려 봅니다. 실패한 유전자 수를 `sum(is.na(...))`로 세어 출력합니다.
5. **traceback 연습.** 아래를 실행해 오류가 난 뒤 콘솔에 `traceback()`을 칩니다. 어느 함수에서 문제가 시작되었는지 읽어 봅니다.

In [ ]:
inner <- function(x) log(x)
outer <- function(v) inner(as.character(v))
outer(1:3)